# Predictive Maintenance for CNC Machines

Predicting machine failure using the [AI4I 2020 Predictive Maintenance Dataset](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) from UCI.

The dataset is heavily imbalanced — only ~3.4% of records are failure events. Most of the work here is handling that imbalance correctly so the model generalizes across failure modes rather than just predicting 'no failure' for everything.

**Approach:** Compare Decision Tree, SVM, and MLP individually, then ensemble them using stacking. Track all experiments with MLflow.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

# Load dataset — download ai4i2020.csv from UCI and place in the same directory
# https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset
data = pd.read_csv('./ai4i2020.csv')

print(f'Dataset shape: {data.shape}')
print(f'Failure rate: {data["Machine failure"].mean():.1%}')  # ~3.4% — heavily imbalanced
print(data['Machine failure'].value_counts())

In [ ]:
# Feature engineering
data_prepared = data.drop(['UDI', 'Product ID'], axis=1)
X = data_prepared.drop('Machine failure', axis=1)
y = data_prepared['Machine failure']

categorical_cols = [c for c in X.columns if X[c].dtype == 'object']
numerical_cols = [c for c in X.columns if X[c].dtype in ['int64', 'float64']]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# SMOTE to handle class imbalance — without this, models predict 'no failure' for everything
# and still get 96% accuracy, which is useless for actual maintenance prediction
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
print(f'After SMOTE:  {dict(pd.Series(y_train_bal).value_counts())}')

In [ ]:
mlflow.set_experiment('predictive_maintenance')

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Neural Network (MLP)': MLPClassifier(max_iter=1000, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42)
}

results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_bal, y_train_bal)
        y_pred = model.predict(X_test)

        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        accuracy = accuracy_score(y_test, y_pred)

        # Log to MLflow
        mlflow.log_param('model', name)
        mlflow.log_metric('precision', precision)
        mlflow.log_metric('recall', recall)
        mlflow.log_metric('f1', f1)
        mlflow.log_metric('accuracy', accuracy)
        mlflow.sklearn.log_model(model, name.replace(' ', '_'))

        results[name] = {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}
        print(f'{name}: Precision={precision:.2f}, Recall={recall:.2f}, F1={f1:.2f}, Accuracy={accuracy:.2f}')

print('\nRun `mlflow ui` to view experiment results at http://localhost:5000')

In [ ]:
# Stacking ensemble — base models feed into a Logistic Regression meta-model
# This generalizes better across different failure modes than any single model
base_models = [
    DecisionTreeClassifier(random_state=42),
    MLPClassifier(max_iter=1000, random_state=42),
    SVC(probability=True, random_state=42)
]

kf = KFold(n_splits=5, random_state=42, shuffle=True)
stacked_train = np.zeros((X_train_bal.shape[0], len(base_models)))

for i, model in enumerate(base_models):
    for train_idx, valid_idx in kf.split(X_train_bal):
        clone_model = clone(model)
        clone_model.fit(X_train_bal[train_idx], y_train_bal.iloc[train_idx])
        stacked_train[valid_idx, i] = clone_model.predict(X_train_bal[valid_idx])

meta_model = LogisticRegression()
meta_model.fit(stacked_train, y_train_bal)

stacked_test = np.zeros((X_test.shape[0], len(base_models)))
for i, model in enumerate(base_models):
    model.fit(X_train_bal, y_train_bal)
    stacked_test[:, i] = model.predict(X_test)

final_predictions = meta_model.predict(stacked_test)

with mlflow.start_run(run_name='Stacking Ensemble'):
    precision = precision_score(y_test, final_predictions)
    recall = recall_score(y_test, final_predictions)
    f1 = f1_score(y_test, final_predictions)
    accuracy = accuracy_score(y_test, final_predictions)

    mlflow.log_param('model', 'Stacking Ensemble')
    mlflow.log_param('base_models', 'DecisionTree + MLP + SVM')
    mlflow.log_param('meta_model', 'LogisticRegression')
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('accuracy', accuracy)

    print('Stacking Ensemble Results:')
    print(f'Precision: {precision:.2f}')
    print(f'Recall:    {recall:.2f}')
    print(f'F1 Score:  {f1:.2f}')
    print(f'Accuracy:  {accuracy:.2f}')
    print()
    print(classification_report(y_test, final_predictions, target_names=['No Failure', 'Failure']))

## Results summary

The key challenge with this dataset is that naive models get ~96% accuracy by predicting 'no failure' for everything — because failures are only 3.4% of records. SMOTE oversampling forces the model to actually learn failure patterns.

The stacking ensemble (Decision Tree + MLP + SVM → Logistic Regression) generalizes best across failure modes (TWF, HDF, PWF, OSF, RNF) because each base model captures different aspects of the failure signal.

All experiments tracked in MLflow — run `mlflow ui` to compare runs.